# Day 32 — Student Performance & Learning Analytics
**Veda Technology · Project 2 of 3**

Explore student academic data, attendance, study habits, subject performance, and introductory predictive modeling.

> This notebook uses synthetic data for learning only.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, classification_report, confusion_matrix)
sns.set_theme(style="whitegrid")

## 2. Generate a Synthetic Student Dataset

In [ ]:
np.random.seed(32)
n = 250
df = pd.DataFrame({
    "student_id": range(1, n + 1),
    "attendance_pct": np.clip(np.random.normal(80, 10, n), 40, 100),
    "study_hours_daily": np.clip(np.random.normal(3.2, 1.4, n), 0.2, 9),
    "assignment_score": np.clip(np.random.normal(73, 12, n), 20, 100),
    "previous_score": np.clip(np.random.normal(68, 14, n), 15, 100),
    "math_score": np.clip(np.random.normal(70, 13, n), 15, 100),
    "science_score": np.clip(np.random.normal(72, 12, n), 15, 100),
    "english_score": np.clip(np.random.normal(74, 11, n), 15, 100)
})
df["final_score"] = np.clip(
    0.22*df.attendance_pct + 2.5*df.study_hours_daily +
    0.20*df.assignment_score + 0.20*df.previous_score +
    0.12*df.math_score + 0.10*df.science_score +
    0.10*df.english_score - 42 + np.random.normal(0, 7, n), 0, 100)
df["passed"] = (df.final_score >= 50).astype(int)
df.head()

## 3. Inspect and Clean Data

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.describe().round(2))
display(df.isna().sum().to_frame("missing_values"))
df = df.drop_duplicates().copy()
num_cols = df.select_dtypes(include="number").columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df.final_score, bins=20, kde=True)
plt.title("Distribution of Final Scores")
plt.show()

plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="attendance_pct", y="final_score")
plt.title("Attendance vs Final Score")
plt.show()

plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="study_hours_daily", y="final_score")
plt.title("Study Hours vs Final Score")
plt.show()

## 5. Subject-Wise Performance

In [ ]:
subjects = ["math_score", "science_score", "english_score"]
means = df[subjects].mean().sort_values(ascending=False)
display(means.round(2).to_frame("average_score"))
means.plot(kind="bar", figsize=(7, 4), title="Average Score by Subject")
plt.ylabel("Average score")
plt.xticks(rotation=0)
plt.show()

## 6. Attendance Analysis

In [ ]:
df["attendance_band"] = pd.cut(
    df.attendance_pct, bins=[0, 60, 75, 90, 100],
    labels=["Below 60%", "60–75%", "75–90%", "90–100%"],
    include_lowest=True)
attendance_summary = df.groupby("attendance_band", observed=False).final_score.agg(
    student_count="count", average_final_score="mean").round(2)
display(attendance_summary)

## 7. Correlation Analysis

In [ ]:
corr = df.select_dtypes("number").drop(columns=["student_id"]).corr()
plt.figure(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()
display(corr.final_score.sort_values(ascending=False).to_frame("correlation_with_final_score"))

## 8. Feature Engineering

In [ ]:
df["average_subject_score"] = df[subjects].mean(axis=1)
df["attendance_study_index"] = df.attendance_pct * df.study_hours_daily
df["assignment_previous_average"] = df[["assignment_score", "previous_score"]].mean(axis=1)
features = ["attendance_pct", "study_hours_daily", "assignment_score", "previous_score",
            "math_score", "science_score", "english_score", "average_subject_score",
            "attendance_study_index", "assignment_previous_average"]
display(df[features + ["final_score", "passed"]].head())

## 9. Linear Regression — Predict Final Score

In [ ]:
X, y = df[features], df.final_score
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=32)
reg = LinearRegression().fit(X_train, y_train)
pred = reg.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 2))
print("R²:", round(r2_score(y_test, pred), 3))

## 10. Logistic Regression — Pass/Fail

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(
    X, df.passed, test_size=0.2, random_state=32, stratify=df.passed)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
clf.fit(Xtr, ytr)
pred_class = clf.predict(Xte)
print("Accuracy:", round(accuracy_score(yte, pred_class), 3))
print(classification_report(yte, pred_class, zero_division=0))
print("Confusion matrix:\\n", confusion_matrix(yte, pred_class))

## 11. Model Interpretation

In [ ]:
coef = pd.Series(reg.coef_, index=features).sort_values(key=abs, ascending=False)
display(coef.to_frame("coefficient"))
coef.sort_values().plot(kind="barh", figsize=(8, 5), title="Regression Coefficients")
plt.show()

## 12. Exploratory Support Flags

In [ ]:
support = df[["student_id", "attendance_pct", "study_hours_daily",
                "average_subject_score", "final_score", "passed"]].copy()
support["exploratory_flag"] = np.select(
    [(support.attendance_pct < 60) | (support.final_score < 40),
     (support.attendance_pct < 75) | (support.final_score < 50)],
    ["Review recommended", "Monitor"], default="No flag from these rules")
display(support.exploratory_flag.value_counts().to_frame("student_count"))
display(support.sort_values("final_score").head(10))

## 13. Summary and Interview Questions

**Workflow:** Data generation, cleaning, EDA, subject and attendance analysis, correlation, feature engineering, regression, classification, and model evaluation.

**Interview questions**
1. What is EDA?
2. How do you handle missing values and duplicates?
3. What is correlation, and why does it not prove causation?
4. What is the difference between Linear and Logistic Regression?
5. Why split data into training and test sets?
6. What do MAE, RMSE, and R² measure?
7. What are precision, recall, and F1-score?
8. Why can feature scaling help Logistic Regression?
9. How would you communicate findings to educators?
10. Why should models not be the sole basis for student decisions?

**Responsible use:** Synthetic data and exploratory flags are for education only, not real student assessment.